# Tecnologías extraordinarias del cine

## Análisis exploratorio de datos

En este notebook analizo 100 tecnologías representadas en el cine para observar cuáles se han convertido en realidad, cuánto tiempo tardaron en aparecer y en qué categorías existe una mayor distancia entre la ficción y la tecnología disponible en 2026.

El análisis servirá como base cuantitativa para una pieza editorial de Nebula. Será un EDA breve, centrado en tres visualizaciones y en conclusiones que puedan explicarse con claridad en el artículo.

In [ ]:
# importo las librerías que voy a utilizar
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# configuro la visualización de tablas y gráficos
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## Carga de los datos

El archivo contiene una observación por tecnología. Una misma película puede aparecer en varias filas cuando presenta más de una innovación relevante.

In [ ]:
# localizo el archivo csv en la carpeta del proyecto
ruta_csv = Path("tecnologias_cine_eda.csv")

# cargo los datos en un dataframe
df = pd.read_csv(ruta_csv)

# reviso las primeras observaciones
df.head()

## Comprobación inicial del dataset

Antes de comenzar el análisis, compruebo el número de observaciones y variables, los tipos de datos, los valores ausentes y la posible existencia de registros duplicados.

Los valores vacíos en `anio_referencia_real` y `anticipacion_anios` no tienen por qué ser errores: aparecen cuando una tecnología todavía no cuenta con una realización comparable.

In [ ]:
# compruebo las dimensiones del dataset
filas, columnas = df.shape
print(f"el dataset contiene {filas} tecnologías y {columnas} variables")

# reviso los tipos de datos de cada variable
display(df.dtypes.to_frame(name="tipo_de_dato"))

# cuento los valores ausentes y los registros duplicados
resumen_calidad = pd.DataFrame({
    "valores_ausentes": df.isna().sum(),
    "porcentaje_ausente": df.isna().mean().mul(100).round(1)
})

display(resumen_calidad)
print(f"registros duplicados: {df.duplicated().sum()}")

## Preparación de las variables

Realizo una preparación mínima antes de representar los datos. Mantengo las valoraciones originales y únicamente ajusto los tipos de datos, ordeno los estados tecnológicos y creo la década de estreno de cada película.

In [ ]:
# convierto los años y la anticipación a enteros que admiten valores ausentes
columnas_enteras = [
    "estreno",
    "anio_referencia_real",
    "anticipacion_anios"
]
df[columnas_enteras] = df[columnas_enteras].astype("Int64")

# defino un orden lógico para los estados tecnológicos
orden_estados = [
    "Ficción",
    "Investigación",
    "Prototipo",
    "Uso real",
    "Cotidiana"
]
df["estado_2026"] = pd.Categorical(
    df["estado_2026"],
    categories=orden_estados,
    ordered=True
)

# creo la década de estreno para facilitar comparaciones temporales
df["decada"] = (df["estreno"] // 10) * 10

# compruebo el resultado de la preparación
df[["pelicula", "estreno", "decada", "estado_2026"]].head()

## 1. ¿Cuántas tecnologías del cine se han hecho realidad?

La primera visualización muestra el grado de desarrollo alcanzado en 2026 por las 100 tecnologías analizadas. Distingo entre tecnologías que continúan siendo ficción, tecnologías en investigación, prototipos, aplicaciones de uso real y soluciones que ya forman parte de la vida cotidiana.

In [ ]:
# calculo el número y el porcentaje de tecnologías en cada estado
resumen_estados = (
    df["estado_2026"]
    .value_counts(sort=False)
    .reindex(orden_estados, fill_value=0)
    .rename_axis("estado_2026")
    .reset_index(name="tecnologias")
)
resumen_estados["porcentaje"] = (
    resumen_estados["tecnologias"] / len(df) * 100
).round(1)

# muestro los valores que voy a representar
display(resumen_estados)

# represento la distribución con un gráfico de barras horizontal
colores_estados = [
    "#D8DDE8",
    "#C9D5ED",
    "#CEC6E8",
    "#B8D8D2",
    "#A9CFB8"
]
fig, ax = plt.subplots(figsize=(9, 4.8))
barras = ax.barh(
    resumen_estados["estado_2026"],
    resumen_estados["porcentaje"],
    color=colores_estados
)

# añado las etiquetas porcentuales al final de cada barra
ax.bar_label(barras, fmt="%.1f %%", padding=4)
ax.set_title("Estado actual de 100 tecnologías imaginadas por el cine", loc="left")
ax.set_xlabel("porcentaje de tecnologías")
ax.set_ylabel("")
ax.set_xlim(0, resumen_estados["porcentaje"].max() + 8)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

## 2. ¿En qué campos se acerca más el cine a la realidad?

Comparo el grado de desarrollo de las tecnologías dentro de cada categoría. Para facilitar la lectura, ordeno las categorías según el porcentaje de tecnologías que ya se encuentran en uso real o cotidiano.

In [ ]:
# calculo la distribución porcentual de estados dentro de cada categoría
estado_por_categoria = pd.crosstab(
    df["categoria"],
    df["estado_2026"],
    normalize="index"
).mul(100)

# mantengo el orden completo de los estados tecnológicos
estado_por_categoria = estado_por_categoria.reindex(
    columns=orden_estados,
    fill_value=0
)

# ordeno las categorías por su proporción de uso real y cotidiano
nivel_realizado = (
    estado_por_categoria["Uso real"]
    + estado_por_categoria["Cotidiana"]
)
orden_categorias = nivel_realizado.sort_values().index
estado_por_categoria = estado_por_categoria.loc[orden_categorias]

# represento la composición de cada categoría con barras apiladas
ax = estado_por_categoria.plot(
    kind="barh",
    stacked=True,
    figsize=(10, 6.5),
    color=colores_estados,
    width=0.72
)

ax.set_title("Grado de desarrollo por categoría tecnológica", loc="left")
ax.set_xlabel("porcentaje dentro de cada categoría")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.legend(
    title="estado en 2026",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

## 3. ¿Qué tecnologías se anticiparon durante más tiempo?

Selecciono las diez tecnologías con mayor distancia entre el estreno de la película y la primera referencia real comparable. El año de referencia no implica una equivalencia completa con la ficción: señala la aparición de una tecnología que comienza a cumplir una función semejante.

Destaco las aportaciones de *Planeta prohibido* para observar su posición dentro del conjunto.

In [ ]:
# selecciono las diez tecnologías con mayor anticipación positiva
ranking_anticipacion = (
    df.loc[df["anticipacion_anios"].gt(0)]
    .nlargest(10, "anticipacion_anios")
    .sort_values("anticipacion_anios")
    .copy()
)

# creo una etiqueta que combina la tecnología y la película
ranking_anticipacion["etiqueta"] = (
    ranking_anticipacion["tecnologia"]
    + " · "
    + ranking_anticipacion["pelicula"]
)

# destaco las tecnologías de planeta prohibido dentro del ranking
colores_ranking = [
    "#A9CFB8" if pelicula == "Planeta prohibido" else "#C9D5ED"
    for pelicula in ranking_anticipacion["pelicula"]
]

# represento los años de anticipación con barras horizontales
fig, ax = plt.subplots(figsize=(10, 6.5))
barras = ax.barh(
    ranking_anticipacion["etiqueta"],
    ranking_anticipacion["anticipacion_anios"],
    color=colores_ranking
)

# añado el número de años al final de cada barra
ax.bar_label(barras, fmt="%d años", padding=4)
ax.set_title("Tecnologías con mayor anticipación cinematográfica", loc="left")
ax.set_xlabel("años entre la película y la referencia real")
ax.set_ylabel("")
ax.set_xlim(0, ranking_anticipacion["anticipacion_anios"].max() + 15)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

## Resultados principales

Resumo las cifras que utilizaré como base para la pieza editorial. Estos resultados describen la muestra seleccionada y no deben interpretarse como una estimación de todo el cine de ciencia ficción.

In [ ]:
# calculo el porcentaje que ya está en uso real o cotidiano
porcentaje_realizado = df["estado_2026"].isin(
    ["Uso real", "Cotidiana"]
).mean() * 100

# calculo el porcentaje que ha alcanzado al menos la fase de prototipo
porcentaje_prototipo = df["estado_2026"].isin(
    ["Prototipo", "Uso real", "Cotidiana"]
).mean() * 100

# identifico la categoría con mayor proporción de tecnologías realizadas
categoria_mas_realizada = nivel_realizado.idxmax()
porcentaje_categoria = nivel_realizado.max()

# cuento las tecnologías de planeta prohibido presentes en el ranking
presencia_planeta_prohibido = (
    ranking_anticipacion["pelicula"] == "Planeta prohibido"
).sum()

# presento las cifras principales en una tabla compacta
resultados_principales = pd.Series({
    "tecnologías en uso real o cotidiano (%)": porcentaje_realizado,
    "tecnologías con al menos un prototipo (%)": porcentaje_prototipo,
    "categoría con mayor realización": categoria_mas_realizada,
    "realización de la categoría líder (%)": porcentaje_categoria,
    "tecnologías de planeta prohibido entre las diez primeras": presencia_planeta_prohibido
})

resultados_principales.to_frame(name="resultado")

## Conclusiones

El 44 % de las tecnologías analizadas ya cuenta con aplicaciones de uso real o cotidiano. Si incorporo los prototipos, el porcentaje asciende al 77 %. La ciencia ficción parece aproximarse mejor a tecnologías relacionadas con la información, la comunicación, la vigilancia y la fabricación que a aquellas que requieren nuevos principios energéticos o físicos.

La categoría con mayor proporción de tecnologías realizadas es **vigilancia, biometría y defensa**, con un 78,6 %. En el extremo contrario, las propuestas clasificadas como **energía y física** permanecen en investigación o continúan siendo ficción.

*Planeta prohibido* ocupa una posición especialmente relevante: tres de sus tecnologías aparecen entre las diez mayores anticipaciones de la muestra. La película se adelantó en la comprensión del lenguaje natural, la robótica multipropósito y el mantenimiento autónomo de infraestructuras. Sin embargo, su tecnología más extraordinaria, la materialización física del subconsciente, continúa fuera de la ciencia conocida.

El resultado sugiere una idea útil para el artículo: el cine suele acertar antes al imaginar **qué función necesitaremos** y **qué problemas puede provocar** que al describir el mecanismo físico con el que llegaremos a construirla.